## 1.Temporal Reviewer Engagement Features (RFM-Inspired)

This notebook engineers RFM-inspired temporal features from the cleaned Silver layer and writes them to `gold.review_temporal_features` in DuckDB. All transforms are implemented as DuckDB window functions over `silver_data`. The final table is also exported as a date-stamped Parquet file under `data/gold/` for downstream ML use.

### 1.1 Motivation

Ngo-Ye & Sinha's early work on reviewer Recency, Frequency, and Monetary Value (RFM) dimensions demonstrated that reviewer engagement profiles significantly improve helpfulness prediction beyond text alone. More recently, **Nayeem & Rafiei (2023)** formalised this in *"On the Role of Reviewer Expertise in Temporal Review Helpfulness Prediction"*, arguing that two features are routinely ignored: (1) *who* posted the review and (2) *when* it was posted relative to the product lifecycle. They show that cold-start reviews (recently submitted, low vote count) are systematically under-predicted by text-only models.

A complementary study — the **temporal regression framework** by Sipos et al. — demonstrates that helpfulness votes are not static: a review's vote count grows non-linearly with its age, meaning raw temporal features without normalisation introduce *sequential bias* (Ocampo et al., as cited in PMC6927604). The implication for our pipeline is that `review_date` is not simply a categorical date field — it encodes *relative temporal position* within the product's review stream that must be computed, not assumed.

On the recency side, **Liu et al. (2022)** found in hospitality reviews that recently posted content receives disproportionately more helpful votes than equally-informative older reviews, confirming that review age itself — rather than just content — is a signal worth engineering explicitly.

### 1.2 Feature Set

| Feature | Description | Rationale |
|---|---|---|
| `review_age_days` | Days between `review_date` and the dataset's max date | Captures absolute age |
| `review_relative_rank` | Chronological position of this review within the product's review stream (1 = first posted) | Captures position bias |
| `product_review_count` | Total reviews per `product_parent` | Proxy for product popularity |
| `days_since_first_review` | Age of product on the market at review time | Product lifecycle position |
| `is_early_review` | Boolean: TRUE if review falls in the earliest 10% for its product | Cold-start indicator |
| `reviews_same_day` | Count of reviews posted on the same date for same product | Review burst detection |
| `review_month`, `review_dayofweek` | Temporal cyclical features | Seasonal/behavioural patterns |  



### References
- Nayeem, M. T., & Rafiei, D. (2023). *On the Role of Reviewer Expertise in Temporal Review Helpfulness Prediction.* arXiv:2303.00923. https://arxiv.org/abs/2303.00923
- Ocampo, J. et al. (cited in PMC6927604). *Feature selection for helpfulness prediction of online product reviews: An empirical study.* PLOS ONE / PMC. https://pmc.ncbi.nlm.nih.gov/articles/PMC6927604/
- Liu, Y. et al. (2022). *Analyzing the impact of review recency on helpfulness through econometric modelling.* International Journal of System Assurance Engineering and Management, Springer. https://doi.org/10.1007/s13198-020-00992-x


In [34]:
# Connect to DuckDB and create the gold schema if it doesn't already exist.
# Close any other DuckDB connections before running — only one write lock allowed.
import duckdb

con = duckdb.connect("../../ProjectData.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

In [ ]:
# Register the cleaned Silver Parquet file as a DuckDB view called silver_data.
# Update the filename here if a newer dated partition has been produced.
con.execute("""
    CREATE OR REPLACE VIEW silver_data AS
    SELECT * FROM read_parquet('../../data/silver/cleaned_data_load_date=2026-03-04.parquet')
""")

In [ ]:
# Quick data check: confirm row count, date range, and number of distinct products.
con.execute("""
    SELECT COUNT(*)                       AS total,
           MIN(review_date)               AS earliest,
           MAX(review_date)               AS latest,
           COUNT(DISTINCT product_parent) AS unique_products
    FROM silver_data
""").df()

In [ ]:
# Verify the per-product review ranking: min rank should always be 1 and max rank
# should equal the product's total review count, confirming correct window ordering.
con.execute("""
    SELECT
        product_parent,
        MIN(review_relative_rank) AS min_rank,
        MAX(review_relative_rank) AS max_rank,
        COUNT(*)                  AS product_review_count_check
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            ) AS review_relative_rank
        FROM silver_data
    )
    GROUP BY product_parent
    ORDER BY product_review_count_check DESC
    LIMIT 10
""").df()

In [ ]:
# Preview all 7 computed temporal features on 20 rows before writing to Gold.
# is_early_review flags reviews in the top 10% earliest per product (cold-start signal).
con.execute("""
    WITH base AS (
        SELECT *,
            MAX(review_date) OVER ()                            AS dataset_max_date,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                   AS review_relative_rank,
            COUNT(*) OVER (PARTITION BY product_parent)         AS product_review_count,
            MIN(review_date) OVER (PARTITION BY product_parent) AS first_review_date,
            COUNT(*) OVER (
                PARTITION BY product_parent, review_date
            )                                                   AS reviews_same_day
        FROM silver_data
    )
    SELECT
        product_parent,
        review_date,
        review_relative_rank,
        product_review_count,
        DATE_DIFF('day', review_date, dataset_max_date)     AS review_age_days,
        DATE_DIFF('day', first_review_date, review_date)    AS days_since_first_review,
        MONTH(review_date)                                  AS review_month,
        DAYOFWEEK(review_date)                              AS review_dayofweek,
        CASE
            WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
            THEN TRUE ELSE FALSE
        END                                                 AS is_early_review
    FROM base
    LIMIT 20
""").df()

In [ ]:
# Sanity checks across the full dataset before writing.
# negative_age and negative_days_since_first must both be 0.
# early_review_count (~52%) is high because many products have only 1–2 reviews,
# so CEIL(10%) = 1 marks every single-review product as early — this is correct.
con.execute("""
    WITH base AS (
        SELECT *,
            MAX(review_date) OVER ()                              AS dataset_max_date,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                     AS review_relative_rank,
            COUNT(*) OVER (PARTITION BY product_parent)           AS product_review_count,
            MIN(review_date) OVER (PARTITION BY product_parent)   AS first_review_date,
            COUNT(*) OVER (
                PARTITION BY product_parent, review_date
            )                                                     AS reviews_same_day
        FROM silver_data
    ),
    ranked AS (
        SELECT *,
            DATE_DIFF('day', review_date, dataset_max_date)       AS review_age_days,
            DATE_DIFF('day', first_review_date, review_date)      AS days_since_first_review,
            MONTH(review_date)                                    AS review_month,
            DAYOFWEEK(review_date)                                AS review_dayofweek,
            CASE
                WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
                THEN TRUE ELSE FALSE
            END                                                   AS is_early_review
        FROM base
    )
    SELECT
        SUM(CASE WHEN review_age_days < 0 THEN 1 ELSE 0 END)         AS negative_age,
        SUM(CASE WHEN days_since_first_review < 0 THEN 1 ELSE 0 END)  AS negative_days_since_first,
        SUM(CASE WHEN is_early_review = TRUE THEN 1 ELSE 0 END)       AS early_review_count,
        AVG(reviews_same_day)                                          AS avg_burst_size,
        MAX(reviews_same_day)                                          AS max_burst_size
    FROM ranked
""").df()

In [ ]:
# Write the full temporal feature set to gold.review_temporal_features.
# CTE 'base' computes all window functions in one pass over silver_data.
# CTE 'ranked' derives date-difference and cyclical features from those results.
con.execute("DROP TABLE IF EXISTS gold.review_temporal_features")
con.execute("""
    CREATE TABLE gold.review_temporal_features AS
    WITH base AS (
        SELECT *,
            MAX(review_date) OVER ()                              AS dataset_max_date,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                     AS review_relative_rank,
            COUNT(*) OVER (PARTITION BY product_parent)           AS product_review_count,
            MIN(review_date) OVER (PARTITION BY product_parent)   AS first_review_date,
            COUNT(*) OVER (
                PARTITION BY product_parent, review_date
            )                                                     AS reviews_same_day
        FROM silver_data
    ),
    ranked AS (
        SELECT *,
            DATE_DIFF('day', review_date, dataset_max_date)       AS review_age_days,
            DATE_DIFF('day', first_review_date, review_date)      AS days_since_first_review,
            MONTH(review_date)                                    AS review_month,
            DAYOFWEEK(review_date)                                AS review_dayofweek,
            CASE
                WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
                THEN TRUE ELSE FALSE
            END                                                   AS is_early_review
        FROM base
    )
    SELECT * FROM ranked
""")
print("gold.review_temporal_features created.")

In [ ]:
# Confirm the Gold table was written correctly: row count, feature distributions,
# and the maximum number of reviews any single product has.
con.execute("""
    SELECT
        COUNT(*)                                      AS total_rows,
        SUM(CASE WHEN is_early_review THEN 1 END)    AS early_reviews,
        ROUND(AVG(review_age_days), 1)               AS avg_age_days,
        ROUND(AVG(days_since_first_review), 1)       AS avg_days_since_first,
        MAX(product_review_count)                    AS max_reviews_per_product
    FROM gold.review_temporal_features
""").df()

In [ ]:
# Export the Gold table to a date-stamped Parquet file for downstream ML use.
from pathlib import Path
from datetime import date

gold_dir = Path("../../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

output_file = gold_dir / f"temporal_features_load_date={date.today().isoformat()}.parquet"
con.execute(f"""
    COPY (SELECT * FROM gold.review_temporal_features)
    TO '{output_file.as_posix()}' (FORMAT PARQUET)
""")
print(f"Saved: {output_file.resolve()}")

In [ ]:
# Release the DuckDB write lock so other processes can connect.
con.close()